In [26]:
"""
1. Run: `pip install openai lancedb tantivy pypdf sqlalchemy phidata cohere` to install the dependencies
2. Run: `python cookbook/rag/03_traditional_rag_lancedb.py` to run the agent
"""

from phi.agent import Agent
from phi.model.groq import  Groq
from phi.embedder.fastembed import FastEmbedEmbedder
from phi.knowledge.pdf import PDFUrlKnowledgeBase
from phi.vectordb.lancedb import LanceDb, SearchType
from phi.embedder.ollama import OllamaEmbedder
from phi.reranker.cohere import CohereReranker
from phi.model.google import  Gemini
from phi.model.openai import OpenAIChat
from dotenv import load_dotenv
import os
load_dotenv()
# openai.api_key=os.getenv("OPENAI_API_KEY")
# api_key=os.getenv("GROQ_API_KEY")
api_key=os.getenv("OPENAI_API_KEY")
hf_key=os.getenv("HUGGINGFACE_API_KEY")
# api_key=os.getenv('GOOGLE_API_KEY')
cohere_key=os.getenv('COHERE_API_KEY')

In [28]:
embedder =OllamaEmbedder(model="nomic-embed-text", dimensions=768)# Ensure this works



In [29]:
knowledge_base = PDFUrlKnowledgeBase(
    urls=["https://phi-public.s3.amazonaws.com/recipes/ThaiRecipes.pdf"],
    # Use LanceDB as the vector database and store embeddings in the `recipes` table
    vector_db=LanceDb(
        table_name="groq_recipes",
        uri="tmp/lancedb",
        search_type=SearchType.vector,
        embedder= embedder,
        reranker=CohereReranker(api_key=cohere_key,model="rerank-multilingual-v3.0"),  # Add a reranker
    ),
)

In [30]:
# Load the knowledge base: Comment after first run as the knowledge base is already loaded
knowledge_base.load()

INFO     Creating collection

INFO     Loading knowledge base

INFO     Reading: https://phi-public.s3.amazonaws.com/recipes/ThaiRecipes.pdf

INFO     Added 14 documents to knowledge base

In [32]:
agent = Agent(
    model=OpenAIChat(id="gpt-4o",api_key=api_key),
    knowledge=knowledge_base,
    # Add a tool to search the knowledge base which enables agentic RAG.
    # This is enabled by default when `knowledge` is provided to the Agent.
    search_knowledge=True,
    show_tool_calls=True,
    markdown=True,
)
agent.print_response("What is the first step of making Gluai Buat Chi from the knowledge base?", stream=True)

In [1]:
from phi.agent import Agent
from phi.model.openai.like import OpenAILike
from phi.embedder.ollama import OllamaEmbedder
from phi.knowledge.pdf import PDFUrlKnowledgeBase
from phi.vectordb.lancedb import LanceDb, SearchType
from phi.reranker.cohere import CohereReranker

c:\Project\PhiData\Financial AI analyst\new_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Create Ollama embedder
embedder = OllamaEmbedder(model="nomic-embed-text", dimensions=768)

In [6]:
# Initialize the OpenAI model with Hyperbolic API
import os
model = OpenAILike(
    id="mixtral-8x7b-32768",  # Specify the model name
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("GROQ_API_KEY")
)


In [3]:
# Create a knowledge base of PDFs from URLs
knowledge_base = PDFUrlKnowledgeBase(
    urls=["https://phi-public.s3.amazonaws.com/recipes/ThaiRecipes.pdf"],
    # Use LanceDB as the vector database and store embeddings in the `recipes` table
    vector_db=LanceDb(
        table_name="recipes",
        uri="tmp/CoherenrankerRag",
        search_type=SearchType.vector,
        embedder=embedder,
        reranker=CohereReranker(model="rerank-multilingual-v3.0"),  # Add a reranker
    ),
)

In [4]:
# Load the knowledge base: Comment after first run as the knowledge base is already loaded
knowledge_base.load()

INFO     Creating collection

INFO     Loading knowledge base

INFO     Reading: https://phi-public.s3.amazonaws.com/recipes/ThaiRecipes.pdf

INFO     Added 14 documents to knowledge base

In [9]:
agent = Agent(
    model=model,
    knowledge=knowledge_base,
    # Add a tool to search the knowledge base which enables agentic RAG.
    # This is enabled by default when `knowledge` is provided to the Agent.
    search_knowledge=True,
    show_tool_calls=True,
)
agent.print_response("How do I make chicken and galangal in coconut milk soup", stream=True)